In [1]:
import os

from IPython.display import FileLink

if os.getcwd() == '/notebooks':
    os.chdir("./motion-synthesis")
    print('inside dir: ', os.listdir())
print(os.listdir())
print("Click here to download the dit_d0.tar: ", display(FileLink("/notebooks/motion-synthesis/dit_d0.tar")))


os.environ["TOKENIZERS_PARALLELISM"] = "false"

['README_v1.md', 'options', 'main.ipynb', 'conda_requirements.txt', '.DS_Store', 'requirements.txt', 'prepare', 'eval_models', 'lambda_requirements.txt', 'data_utils', 'checkpoints', 'utils', 'README.md', 'dataset_split_builder.py', 'networks', 'common', 'glove', '.gitignore', 'log', 'environment_cpu.yaml', 'environment.yaml', 'exp_results', '.git', '.vscode', 'main.py', 'demo.py', 'data', 'assets', 'outputs']


/notebooks/motion-synthesis/dit_d0.tar

Click here to download the dit_d0.tar:  None


In [2]:
import torch
from torch.backends import cuda
from options.train_options  import TrainOptions
from os.path import join as pjoin
import os
from utils.paramUtils import t2m_kinematic_chain
import numpy as np
from utils.word_vectorizer import WordVectorizer
from torch.utils.data import DataLoader
from data_utils.dataset import MotionDataset
from data_utils.dataset import PartMotionDataset
from networks.nn import MotionVQVAE, DiT, MotionVAE
from networks.trainers import MotionVQVAETrainer, MotionDiTTrainer, MotionVAETrainer
from torch.utils.data import Subset
from networks.nn_validator import VAEValidator, DiffusionValidator
import json

/Users/beethoven/miniforge3/envs/vper-motion-editing/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
parser = TrainOptions()
options = parser.parse(args = ['--stage', 'autoencoder', '--max_epoch', '50', '--kl_beta_max','0.001', '--kl_beta_cycles', '3', '--lr', '1e-4', '--save_latest', '50', '--eval_every_e', '1', '--save_every_e', '50', '--log_every', '1'])
options.gpu_id = torch.cuda.current_device() if torch.cuda.is_available() else -1
options.device = torch.device("cpu" if options.gpu_id==-1 else "cuda:" + str(options.gpu_id))
torch.autograd.set_detect_anomaly(True)

# disabling flash backend till the architecture parameters allow stable use of flash backend for attention
# without creating nans
cuda.enable_flash_sdp(False)
cuda.enable_mem_efficient_sdp(False)
cuda.enable_math_sdp(True)

if options.gpu_id != -1:
    # self.opt.gpu_id = int(self.opt.gpu_id)
    torch.cuda.set_device(options.gpu_id)

print('\nDevice used: ', options.device)
options.save_root = pjoin(options.checkpoints_dir, 'HumanML3D', options.name)
options.model_dir = pjoin(options.checkpoints_dir, 'model')
options.meta_dir = pjoin(options.save_root, 'meta')
options.eval_dir = pjoin(options.save_root, 'animation')
options.log_dir = pjoin('./log', options.dataset_name, options.name)
options.experiment_dir = './exp_results/vae-setup/finetune_vae'
#options.experiment_dir = './exp_results/stable-diffusion-setup/pipeline_1a_clip_tsg_ca'
options.output_dir = options.experiment_dir
options.is_train = True
options.is_continue = False
options.dataset_mode = "micro"
options.batch_size = 128
#options.model_filename = 'dit_stable_crossattn_full.tar'
options.model_filename = 'motionvae_full.tar'
os.makedirs(options.model_dir, exist_ok=True)
os.makedirs(options.meta_dir, exist_ok=True)
os.makedirs(options.eval_dir, exist_ok=True)
os.makedirs(options.log_dir, exist_ok=True)

options.data_root = './data/HumanML3D'
options.motion_dir = pjoin(options.data_root, 'new_joint_vecs')
options.text_dir = pjoin(options.data_root, 'texts')
options.joints_num = 22
options.max_motion_length = 120
dim_pose = 263
radius = 4
fps = 20
kinematic_chain = t2m_kinematic_chain

print('Set parameters')
print('Learning rate: ', options.lr)
print('Max epochs: ', options.max_epoch)
print('Save latest frequency: ', options.save_latest)
print('Save every epoch frequency: ', options.save_every_e)
print('Log every iterations frequency: ', options.log_every)


Device used:  cpu
Set parameters
Learning rate:  0.0001
Max epochs:  50
Save latest frequency:  50
Save every epoch frequency:  50
Log every iterations frequency:  1


In [4]:
mean = np.load(pjoin(options.data_root, 'Mean.npy'))
std = np.load(pjoin(options.data_root, 'Std.npy'))

w_vectorizer = WordVectorizer('./glove', 'our_vab')
train_split_fn = 'train.txt'
val_split_fn = 'val.txt'
test_split_fn = 'test.txt'
simple_test_split_fn = 'simple_test.txt'

if options.dataset_mode in ["debug", "nano", "micro"]:
    train_split_fn = f'train_{options.dataset_mode}.txt'
    val_split_fn = f'val_{options.dataset_mode}.txt'

train_split_file = pjoin(options.data_root, train_split_fn)
val_split_file = pjoin(options.data_root, val_split_fn)
test_split_file = pjoin(options.data_root, test_split_fn)
simple_test_split_file = pjoin(options.data_root, simple_test_split_fn)

train_dataset = PartMotionDataset(options, mean, std, train_split_file, w_vectorizer)
val_dataset = PartMotionDataset(options, mean, std, val_split_file, w_vectorizer)
if not(options.is_train):
    test_dataset = PartMotionDataset(options, mean, std, test_split_file, w_vectorizer)

print('\nTotal number of snippets in train: ', len(train_dataset))
print('Total number of snippets in val: ', len(val_dataset))
if not(options.is_train):
    print('Total number of snippets in test: ', len(test_dataset))

sample_motion = train_dataset[4]
print('Sample data shape: ', sample_motion['motion_parts'].shape, sample_motion['text'])
Dp_max = sample_motion['motion_parts'].shape[-1]

id list 128


100%|██████████| 128/128 [00:01<00:00, 91.71it/s]


Motion shape (B, T, D): (382, 120, 263)
Total number of motions 382
Total number of small motions: 61
id list 64


100%|██████████| 64/64 [00:00<00:00, 88.14it/s]

Motion shape (B, T, D): (191, 120, 263)
Total number of motions 191
Total number of small motions: 26

Total number of snippets in train:  382
Total number of snippets in val:  191
Sample data shape:  (120, 6, 60) the person is jumping up and down.


In [5]:
train_loader = DataLoader(train_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode in ['micro', 'nano']), num_workers=1,
                              shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode in ['micro', 'nano']), num_workers=1,
                        shuffle=True, pin_memory=True)
if not(options.is_train):
    test_loader = DataLoader(test_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode in ['micro', 'nano']), num_workers=1,
                             shuffle=True, pin_memory=True)

if options.stage == "autoencoder":
    '''
    vqvae = MotionVQVAE(
        input_dim=Dp_max,
        enc_hidden_dim=1024,
        dec_hidden_dim=1024,
        latent_dim=256,
        num_embeddings=512,
        beta=0.1
    )

    if options.is_train:
        trainer = MotionVQVAETrainer(options, vqvae = vqvae)
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader)
    '''
    vae = MotionVAE(
        dim = 263, #input dimension of motion vector
        hidden_size = 512, # latent dimension
        max_seq_len=options.max_motion_length,
        num_heads = 4,
        depth = 9
    )
    if options.is_train:
        trainer = MotionVAETrainer(options, vae = vae)
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader
        )
else:
    dit = DiT(
        input_size = 512, # latent dimension
        hidden_size = 1152,
        text_dim = 768,
        max_seq_len=options.max_motion_length // 4
    )

    if options.is_train:
        trainer = MotionDiTTrainer(
            args = options,
            dit = dit,
            autoencoder_type="pretrained_vae"
        )
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader
        )


Number of epochs: 50
Iters Per Epoch, Training: 0003, Validation: 002


/Users/beethoven/miniforge3/envs/vper-motion-editing/lib/python3.11/site-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


x shape after pos enc: torch.Size([128, 30, 512]) torch.Size([128, 120, 512])
qkv shapes: torch.Size([128, 4, 30, 128]) torch.Size([128, 4, 120, 128]) torch.Size([128, 4, 120, 128])
x shape after pos enc: torch.Size([128, 30, 512]) torch.Size([128, 120, 512])
qkv shapes: torch.Size([128, 4, 30, 128]) torch.Size([128, 4, 120, 128]) torch.Size([128, 4, 120, 128])
x shape after pos enc: torch.Size([126, 30, 512]) torch.Size([126, 120, 512])
qkv shapes: torch.Size([126, 4, 30, 128]) torch.Size([126, 4, 120, 128]) torch.Size([126, 4, 120, 128])
x shape after pos enc: torch.Size([128, 30, 512]) torch.Size([128, 120, 512])
qkv shapes: torch.Size([128, 4, 30, 128]) torch.Size([128, 4, 120, 128]) torch.Size([128, 4, 120, 128])
x shape after pos enc: torch.Size([63, 30, 512]) torch.Size([63, 120, 512])
qkv shapes: torch.Size([63, 4, 30, 128]) torch.Size([63, 4, 120, 128]) torch.Size([63, 4, 120, 128])
Epoch: 0
Train Loss: 1.34826 Reconstruction Loss: 1.29280 KL Loss: 296.77380
Validation Loss: 1

KeyboardInterrupt: 

In [ ]:
if options.stage == "autoencoder":
    test_model_filepath = pjoin(options.model_dir, options.model_filename)

    if not(options.is_train) and os.path.exists(test_model_filepath):
        vae_model_dict = torch.load(test_model_filepath, map_location = options.device)

        vae.load_state_dict(vae_model_dict['best_vae'])

        vae_validator = VAEValidator(
            opt = options,
            vae_model=vae,
            model_type = "vae",
            train_dataloader=train_loader,
            val_dataloader = val_loader
        )
        vae_validator.validate()
    else:
        print("Invalid mode or model file doesn't exist!")
else:
    test_model_filepath = pjoin(options.model_dir, options.model_filename)
    if not(options.is_train) and os.path.exists(test_model_filepath):
        dit_model_dict = torch.load(test_model_filepath, map_location = options.device)

        dit.load_state_dict(dit_model_dict['dit'])

        dit_validator = DiffusionValidator(
            opt = options,
            dit=dit,
            val_dataloader = val_loader,
            test_dataloader=test_loader,
            test_type = "val"
        )
        dit_validator.validate()
    else:
        print("Invalid mode or model file doesn't exist!")

/Users/beethoven/miniforge3/envs/vper-motion-editing/lib/python3.11/site-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


mask shape:  torch.Size([1, 120])
full motion recon:  (1, 120, 263)
Saved video file at:  ./exp_results/vae-setup/finetune_vae/videos/train_011519_70.gif
mask shape:  torch.Size([1, 120])
full motion recon:  (1, 120, 263)
Saved video file at:  ./exp_results/vae-setup/finetune_vae/videos/train_004460_62.gif
mask shape:  torch.Size([1, 120])
full motion recon:  (1, 120, 263)
Saved video file at:  ./exp_results/vae-setup/finetune_vae/videos/train_M011586_28.gif


/Users/beethoven/miniforge3/envs/vper-motion-editing/lib/python3.11/site-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


mask shape:  torch.Size([1, 120])
full motion recon:  (1, 120, 263)
Saved video file at:  ./exp_results/vae-setup/finetune_vae/videos/val_000646_70.gif
mask shape:  torch.Size([1, 120])
full motion recon:  (1, 120, 263)
Saved video file at:  ./exp_results/vae-setup/finetune_vae/videos/val_011384_15.gif
